## Numpy array and basics

In [1]:
import numpy as np

In [2]:
print(globals())

{'__name__': '__main__', '__doc__': 'Automatically created module for IPython interactive environment', '__package__': None, '__loader__': None, '__spec__': None, '__builtin__': <module 'builtins' (built-in)>, '__builtins__': <module 'builtins' (built-in)>, '_ih': ['', 'import numpy as np', 'print(globals())'], '_oh': {}, '_dh': [PosixPath('/home/ndey/Python/Numpy')], 'In': ['', 'import numpy as np', 'print(globals())'], 'Out': {}, 'get_ipython': <bound method InteractiveShell.get_ipython of <ipykernel.zmqshell.ZMQInteractiveShell object at 0x7c524dea0860>>, 'exit': <IPython.core.autocall.ZMQExitAutocall object at 0x7c524c12be60>, 'quit': <IPython.core.autocall.ZMQExitAutocall object at 0x7c524c12be60>, 'open': <function open at 0x7c524e9bd3a0>, '_': '', '__': '', '___': '', '__vsc_ipynb_file__': '/home/ndey/Python/Numpy/phase1.ipynb', '_i': 'import numpy as np', '_ii': '', '_iii': '', '_i1': 'import numpy as np', 'np': <module 'numpy' from '/home/ndey/Python/Numpy/.venv/lib/python3.12

list and numpy array memoy management

In [3]:
import sys
import time
import numpy as np

# ============================================================
# CREATE TEST DATA
# ============================================================
# 1 million integers
N = 1_000_000

# Python List
# Stores references (pointers) to Python int objects
ls = list(range(N))

# NumPy Array
# Stores actual values in one contiguous memory block
arr = np.array(ls, dtype=np.int64)

# ============================================================
# MEMORY COMPARISON
# ============================================================

print("\n" + "=" * 60)
print("MEMORY COMPARISON")
print("=" * 60)

# Size of list object itself
# (does NOT include memory of all integer objects)
print("List object size :", sys.getsizeof(ls), "bytes")

# Size of NumPy array object + buffer metadata
print("NumPy object size:", sys.getsizeof(arr), "bytes")

# Actual memory occupied by array data
print("NumPy raw data   :", arr.nbytes, "bytes")

# Starting address of contiguous memory block
print("\nNumPy Base Address:", hex(arr.ctypes.data))

# Size of each element
print("Element Size      :", arr.itemsize, "bytes")

# Verify array is contiguous
print("C_CONTIGUOUS      :", arr.flags['C_CONTIGUOUS'])

# ============================================================
# SHOW MEMORY ADDRESSES
# ============================================================

print("\n" + "=" * 60)
print("FIRST 10 LIST ELEMENT IDS")
print("=" * 60)

for i in range(10):
    # Address of Python integer object
    print(f"ls[{i}] = {ls[i]} -> {hex(id(ls[i]))}")

print("\n" + "=" * 60)
print("FIRST 10 NUMPY ELEMENT ADDRESSES")
print("=" * 60)

for i in range(10):

    # Actual address of ith element in contiguous block
    address = arr.ctypes.data + (i * arr.itemsize)

    print(f"arr[{i}] = {arr[i]} -> {hex(address)}")

# Notice:
#
# List IDs may not be sequential.
# NumPy addresses increase by exactly 8 bytes (int64).
#
# Example:
#
# arr[0] -> 0x1000
# arr[1] -> 0x1008
# arr[2] -> 0x1010
#
# This proves contiguous storage.


# ============================================================
# BENCHMARK 1
# LOOPING THROUGH PYTHON LIST
# ============================================================

print("\n" + "=" * 60)
print("PYTHON LOOP OVER LIST")
print("=" * 60)

start = time.perf_counter()

total = 0

for x in ls:
    total += x

list_loop_time = time.perf_counter() - start

print("Total:", total)
print("Time :", list_loop_time)


# ============================================================
# BENCHMARK 2
# LOOPING THROUGH NUMPY ARRAY
# ============================================================

print("\n" + "=" * 60)
print("PYTHON LOOP OVER NUMPY ARRAY")
print("=" * 60)

start = time.perf_counter()

total = 0

for x in arr:
    total += x

numpy_loop_time = time.perf_counter() - start

print("Total:", total)
print("Time :", numpy_loop_time)

# ------------------------------------------------------------
# Why NumPy may NOT be faster here?
#
# Python executes the loop.
#
# For every iteration:
#
# NumPy must:
# 1. Read value from contiguous memory.
# 2. Create a numpy.int64 scalar object.
# 3. Return it to Python.
#
# So Python still controls the loop.
#
# Result:
# List loop is often similar or faster.
# ------------------------------------------------------------


# ============================================================
# BENCHMARK 3
# NUMPY SUM
# ============================================================

print("\n" + "=" * 60)
print("NUMPY SUM")
print("=" * 60)

start = time.perf_counter()

# Entire operation happens in optimized C code
total = arr.sum()

numpy_sum_time = time.perf_counter() - start

print("Total:", total)
print("Time :", numpy_sum_time)

# ------------------------------------------------------------
# Why is this fast?
#
# No Python loop.
#
# NumPy directly processes contiguous memory using C.
#
# CPU cache utilization is excellent because
# values are stored next to each other.
# ------------------------------------------------------------


# ============================================================
# BENCHMARK 4
# LIST MULTIPLICATION
# ============================================================

print("\n" + "=" * 60)
print("LIST COMPREHENSION")
print("=" * 60)

start = time.perf_counter()

result = [x * 2 for x in ls]

list_mult_time = time.perf_counter() - start

print("Time :", list_mult_time)

# ------------------------------------------------------------
# Python performs:
#
# x * 2
#
# one element at a time.
#
# Millions of Python-level operations.
# ------------------------------------------------------------


# ============================================================
# BENCHMARK 5
# NUMPY VECTORIZED MULTIPLICATION
# ============================================================

print("\n" + "=" * 60)
print("NUMPY VECTORIZED MULTIPLICATION")
print("=" * 60)

start = time.perf_counter()

result = arr * 2

numpy_mult_time = time.perf_counter() - start

print("Time :", numpy_mult_time)

# ------------------------------------------------------------
# Why much faster?
#
# NumPy executes:
#
# arr * 2
#
# entirely in compiled C code.
#
# Memory is contiguous:
#
# +----+----+----+----+
# | 1  | 2  | 3  | 4  |
# +----+----+----+----+
#
# CPU reads sequential memory efficiently.
#
# No Python loop overhead.
# ------------------------------------------------------------


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"List Loop                 : {list_loop_time:.6f} sec")
print(f"NumPy Loop                : {numpy_loop_time:.6f} sec")
print(f"NumPy Sum                 : {numpy_sum_time:.6f} sec")
print(f"List Comprehension (*2)   : {list_mult_time:.6f} sec")
print(f"NumPy Vectorized (*2)     : {numpy_mult_time:.6f} sec")

# ============================================================
# CONCLUSION
# ============================================================
#
# Python List:
#   ✓ Good for general-purpose programming.
#   ✓ Flexible (can store mixed data types).
#   ✗ Not ideal for heavy numerical computation.
#
# NumPy Array:
#   ✓ Stores homogeneous data contiguously.
#   ✓ Excellent cache locality.
#   ✓ Vectorized operations run in C.
#   ✓ Much faster for mathematical operations.
#
# IMPORTANT:
#
# NumPy is NOT faster because "for loop is faster".
#
# NumPy is faster because:
#
#     arr.sum()
#     arr * 2
#     arr + arr
#     matrix multiplication
#
# are executed in optimized C code on contiguous memory.
#
# That's where the real performance gain comes from.
# ============================================================


MEMORY COMPARISON
List object size : 8000056 bytes
NumPy object size: 8000112 bytes
NumPy raw data   : 8000000 bytes

NumPy Base Address: 0x7c520aab7010
Element Size      : 8 bytes
C_CONTIGUOUS      : True

FIRST 10 LIST ELEMENT IDS
ls[0] = 0 -> 0xb36088
ls[1] = 1 -> 0xb360a8
ls[2] = 2 -> 0xb360c8
ls[3] = 3 -> 0xb360e8
ls[4] = 4 -> 0xb36108
ls[5] = 5 -> 0xb36128
ls[6] = 6 -> 0xb36148
ls[7] = 7 -> 0xb36168
ls[8] = 8 -> 0xb36188
ls[9] = 9 -> 0xb361a8

FIRST 10 NUMPY ELEMENT ADDRESSES
arr[0] = 0 -> 0x7c520aab7010
arr[1] = 1 -> 0x7c520aab7018
arr[2] = 2 -> 0x7c520aab7020
arr[3] = 3 -> 0x7c520aab7028
arr[4] = 4 -> 0x7c520aab7030
arr[5] = 5 -> 0x7c520aab7038
arr[6] = 6 -> 0x7c520aab7040
arr[7] = 7 -> 0x7c520aab7048
arr[8] = 8 -> 0x7c520aab7050
arr[9] = 9 -> 0x7c520aab7058

PYTHON LOOP OVER LIST
Total: 499999500000
Time : 0.08690112700060126

PYTHON LOOP OVER NUMPY ARRAY
Total: 499999500000
Time : 0.12633418699988397

NUMPY SUM
Total: 499999500000
Time : 0.00888213399957749

LIST COMPREHENSI

creating array from list

In [4]:
ls = [1,7,8,9,10]
arr_1d = np.array(ls)

print(type(arr_1d))
arr_1d = np.arange(1,10,2)
print(arr_1d)
arr_1d = np.random.random(10)
print(arr_1d)
arr_1d = np.zeros(7)
print(arr_1d)
arr_1d = np.ones(7)
print(arr_1d)
arr_1d = np.full(4,7)
print(arr_1d)



<class 'numpy.ndarray'>
[1 3 5 7 9]
[0.49087776 0.04750997 0.85241748 0.13516644 0.15125391 0.33530058
 0.881495   0.59694601 0.37286452 0.43410081]
[0. 0. 0. 0. 0. 0. 0.]
[1. 1. 1. 1. 1. 1. 1.]
[7 7 7 7]


In [5]:
print('__iter__' in dir(arr_1d))
print('__next__' in dir(arr_1d))

# numpy array's are iteratble object

True
False


2D array / matrix

In [6]:
arr_2d = np.array([[1,2,3],[4,5,6],[7,8,9]])
print(arr_2d)
print(type(arr_2d))
arr_2d = np.zeros((3,3))
print(arr_2d)
arr_2d = np.ones((2,3))
print(arr_2d)
arr_2d = np.full((2,3),5)
print(arr_2d)

#identity matrices
arr_2d = np.eye(4)
print(arr_2d)

[[1 2 3]
 [4 5 6]
 [7 8 9]]
<class 'numpy.ndarray'>
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[1. 1. 1.]
 [1. 1. 1.]]
[[5 5 5]
 [5 5 5]]
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


Properties

In [7]:
print(arr_2d.shape)
print(arr_2d.size)
print(arr_2d.ndim)
print(arr_2d.dtype)
arr2d = arr_2d.astype(str)
print(arr2d.dtype)
print(arr2d)


print(arr_1d.shape)
print(arr_1d.size)
print(arr_1d.ndim)
print(arr_1d.dtype)


(4, 4)
16
2
float64
<U32
[['1.0' '0.0' '0.0' '0.0']
 ['0.0' '1.0' '0.0' '0.0']
 ['0.0' '0.0' '1.0' '0.0']
 ['0.0' '0.0' '0.0' '1.0']]
(4,)
4
1
int64


Mathematical operation

In [8]:
print(arr_2d + 5)
print(arr_2d * 6)
print(arr_2d // 7)
print(arr_2d % 7)

[[6. 5. 5. 5.]
 [5. 6. 5. 5.]
 [5. 5. 6. 5.]
 [5. 5. 5. 6.]]
[[6. 0. 0. 0.]
 [0. 6. 0. 0.]
 [0. 0. 6. 0.]
 [0. 0. 0. 6.]]
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


In [9]:
arrr = np.random.random((5,4))*100
print(arrr)
print(np.max(arrr))
print(np.max(arrr[:3],axis = 0))
print()
print(np.min(arrr))
print(np.sum(arrr, axis = 0))
print(np.mean(arrr))
print(np.std(arrr))
print(np.var(arrr))



[[46.58060218 40.90810748 32.52279409 90.92355161]
 [83.65785074  5.41468314 84.48065195  6.24420909]
 [33.39425496 74.08322755 32.75969266 93.85287727]
 [53.85698878 60.24326808 45.86371321 59.56819931]
 [61.60810877 22.54778215 12.96277473 67.37056187]]
93.8528772736624
[83.65785074 74.08322755 84.48065195 93.85287727]

5.414683144798138
[279.09780542 203.1970684  208.58962664 317.95939916]
50.44219498108734
26.609397454903696
708.0600329130352


Indexing & slicing

In [10]:
arr_3d = np.random.random((3,3,2))*100
print(arr_3d)
print(arr_3d[0][0][1])

print(arr_3d[(0,0,1)])
#fancy indexing
print(arr_3d[[0,0,1]])



[[[74.57194573 26.6287272 ]
  [25.89051119 14.6253399 ]
  [26.04900269 29.92827559]]

 [[55.63977819 97.96442508]
  [56.07020834 93.01328392]
  [68.01925145 27.60603819]]

 [[57.98974929 22.00073098]
  [33.28478876 19.41190958]
  [80.65706411 53.74191979]]]
26.628727200410584
26.628727200410584
[[[74.57194573 26.6287272 ]
  [25.89051119 14.6253399 ]
  [26.04900269 29.92827559]]

 [[74.57194573 26.6287272 ]
  [25.89051119 14.6253399 ]
  [26.04900269 29.92827559]]

 [[55.63977819 97.96442508]
  [56.07020834 93.01328392]
  [68.01925145 27.60603819]]]


In [11]:
print(arr_3d[:,0,:1])
print(arr_3d[:,0,0])
print(arr_3d[:,0])
print(arr_3d[:,:1])

print(arr_3d[:,:,0])

[[74.57194573]
 [55.63977819]
 [57.98974929]]
[74.57194573 55.63977819 57.98974929]
[[74.57194573 26.6287272 ]
 [55.63977819 97.96442508]
 [57.98974929 22.00073098]]
[[[74.57194573 26.6287272 ]]

 [[55.63977819 97.96442508]]

 [[57.98974929 22.00073098]]]
[[74.57194573 25.89051119 26.04900269]
 [55.63977819 56.07020834 68.01925145]
 [57.98974929 33.28478876 80.65706411]]


Filtering

In [12]:
print(arr_3d[arr_3d>5])


[74.57194573 26.6287272  25.89051119 14.6253399  26.04900269 29.92827559
 55.63977819 97.96442508 56.07020834 93.01328392 68.01925145 27.60603819
 57.98974929 22.00073098 33.28478876 19.41190958 80.65706411 53.74191979]


Fast Conditional Formatting

In [13]:
arr_3de = np.astype(arr_3d,int)
print(arr_3de)

print(np.where(arr_3de>50))

print(arr_3de[np.where(arr_3de>50)])

print(np.where(arr_3de>50, np.nan , arr_3de))


[[[74 26]
  [25 14]
  [26 29]]

 [[55 97]
  [56 93]
  [68 27]]

 [[57 22]
  [33 19]
  [80 53]]]
(array([0, 1, 1, 1, 1, 1, 2, 2, 2]), array([0, 0, 0, 1, 1, 2, 0, 2, 2]), array([0, 0, 1, 0, 1, 0, 0, 0, 1]))
[74 55 97 56 93 68 57 80 53]
[[[nan 26.]
  [25. 14.]
  [26. 29.]]

 [[nan nan]
  [nan nan]
  [nan 27.]]

 [[nan 22.]
  [33. 19.]
  [nan nan]]]


Reshapping

In [14]:
print(arr_3d)

print()
print(arr_3d.reshape(9,2))

print()
print(arr_3d.reshape(6,3))

print()
print(arr_3d.flatten())

print()
print(arr_3d.ravel())
#ravel creates a view of actual object, and if you change on view actual object will also change

print(arr_3d)

[[[74.57194573 26.6287272 ]
  [25.89051119 14.6253399 ]
  [26.04900269 29.92827559]]

 [[55.63977819 97.96442508]
  [56.07020834 93.01328392]
  [68.01925145 27.60603819]]

 [[57.98974929 22.00073098]
  [33.28478876 19.41190958]
  [80.65706411 53.74191979]]]

[[74.57194573 26.6287272 ]
 [25.89051119 14.6253399 ]
 [26.04900269 29.92827559]
 [55.63977819 97.96442508]
 [56.07020834 93.01328392]
 [68.01925145 27.60603819]
 [57.98974929 22.00073098]
 [33.28478876 19.41190958]
 [80.65706411 53.74191979]]

[[74.57194573 26.6287272  25.89051119]
 [14.6253399  26.04900269 29.92827559]
 [55.63977819 97.96442508 56.07020834]
 [93.01328392 68.01925145 27.60603819]
 [57.98974929 22.00073098 33.28478876]
 [19.41190958 80.65706411 53.74191979]]

[74.57194573 26.6287272  25.89051119 14.6253399  26.04900269 29.92827559
 55.63977819 97.96442508 56.07020834 93.01328392 68.01925145 27.60603819
 57.98974929 22.00073098 33.28478876 19.41190958 80.65706411 53.74191979]

[74.57194573 26.6287272  25.89051119 14

Insertion

In [15]:
print(arr_3d)

print(np.insert(arr_3d,0,[80,9])) # by default axis = None, means flattered version
print(arr_3d.shape)

print(np.insert(arr_3d,1,[[80,89],[87,88],[99,78]],axis = 1))
print(np.insert(arr_3d,1,[[80,89,90],[87,88,87],[99,78,78]],axis = 2))

# delete also works same




[[[74.57194573 26.6287272 ]
  [25.89051119 14.6253399 ]
  [26.04900269 29.92827559]]

 [[55.63977819 97.96442508]
  [56.07020834 93.01328392]
  [68.01925145 27.60603819]]

 [[57.98974929 22.00073098]
  [33.28478876 19.41190958]
  [80.65706411 53.74191979]]]
[80.          9.         74.57194573 26.6287272  25.89051119 14.6253399
 26.04900269 29.92827559 55.63977819 97.96442508 56.07020834 93.01328392
 68.01925145 27.60603819 57.98974929 22.00073098 33.28478876 19.41190958
 80.65706411 53.74191979]
(3, 3, 2)
[[[74.57194573 26.6287272 ]
  [80.         89.        ]
  [25.89051119 14.6253399 ]
  [26.04900269 29.92827559]]

 [[55.63977819 97.96442508]
  [87.         88.        ]
  [56.07020834 93.01328392]
  [68.01925145 27.60603819]]

 [[57.98974929 22.00073098]
  [99.         78.        ]
  [33.28478876 19.41190958]
  [80.65706411 53.74191979]]]
[[[74.57194573 80.         26.6287272 ]
  [25.89051119 89.         14.6253399 ]
  [26.04900269 90.         29.92827559]]

 [[55.63977819 87.      

Concatination

In [16]:
arr_3d1 = np.random.random((3,2,3))*10
print(arr_3d1)
arr_3d2 = np.random.random((3,2,3))*10
print()
print(arr_3d2)
print()
print(np.concatenate((arr_3d1,arr_3d2),axis = None))
print()
print(np.concatenate((arr_3d1,arr_3d2),axis = 0))
print()

print(np.concatenate((arr_3d1,arr_3d2),axis = 1))
print()

print(np.concatenate((arr_3d1,arr_3d2),axis = 2))

[[[1.21341107 6.16153957 6.27903537]
  [8.63654602 4.28806146 0.15567044]]

 [[5.88864478 0.33310596 4.36183987]
  [0.44170854 0.30828904 9.60493547]]

 [[1.30317432 3.60150191 4.24833298]
  [0.27268215 8.90799265 9.30143289]]]

[[[8.98188787 3.75093366 8.05512483]
  [9.43907437 0.924208   5.86009481]]

 [[6.57690663 6.12033014 9.28306021]
  [0.2280657  9.82400165 7.87082105]]

 [[8.87156064 2.49329706 6.92551859]
  [2.16552682 8.457417   2.41812215]]]

[1.21341107 6.16153957 6.27903537 8.63654602 4.28806146 0.15567044
 5.88864478 0.33310596 4.36183987 0.44170854 0.30828904 9.60493547
 1.30317432 3.60150191 4.24833298 0.27268215 8.90799265 9.30143289
 8.98188787 3.75093366 8.05512483 9.43907437 0.924208   5.86009481
 6.57690663 6.12033014 9.28306021 0.2280657  9.82400165 7.87082105
 8.87156064 2.49329706 6.92551859 2.16552682 8.457417   2.41812215]

[[[1.21341107 6.16153957 6.27903537]
  [8.63654602 4.28806146 0.15567044]]

 [[5.88864478 0.33310596 4.36183987]
  [0.44170854 0.30828904 

In [17]:
ad = np.array([[2,3],[4,6]])
print(np.insert(ad,2,[7,8],axis = 0))

print(np.vstack((ad,[[7,8]])))
print(np.insert(ad,2,[[7,9],[8,10]],axis = 1))
print(np.hstack((ad,[[7],[8]])))

# for concatinaton, always the dimension except the axis of concatination should of same shape.


[[2 3]
 [4 6]
 [7 8]]
[[2 3]
 [4 6]
 [7 8]]
[[ 2  3  7  8]
 [ 4  6  9 10]]
[[2 3 7]
 [4 6 8]]


Splitting

In [18]:
print(arr_3d)
print(np.split(arr_3d,3))
print(np.split(arr_3d,2,axis = 2))


[[[74.57194573 26.6287272 ]
  [25.89051119 14.6253399 ]
  [26.04900269 29.92827559]]

 [[55.63977819 97.96442508]
  [56.07020834 93.01328392]
  [68.01925145 27.60603819]]

 [[57.98974929 22.00073098]
  [33.28478876 19.41190958]
  [80.65706411 53.74191979]]]
[array([[[74.57194573, 26.6287272 ],
        [25.89051119, 14.6253399 ],
        [26.04900269, 29.92827559]]]), array([[[55.63977819, 97.96442508],
        [56.07020834, 93.01328392],
        [68.01925145, 27.60603819]]]), array([[[57.98974929, 22.00073098],
        [33.28478876, 19.41190958],
        [80.65706411, 53.74191979]]])]
[array([[[74.57194573],
        [25.89051119],
        [26.04900269]],

       [[55.63977819],
        [56.07020834],
        [68.01925145]],

       [[57.98974929],
        [33.28478876],
        [80.65706411]]]), array([[[26.6287272 ],
        [14.6253399 ],
        [29.92827559]],

       [[97.96442508],
        [93.01328392],
        [27.60603819]],

       [[22.00073098],
        [19.41190958],
     

Vectorized Operations

In [19]:
vector1 = np.array([2,3,7])
vector2 = np.array([8,9,5])

print("vector addition :- \n",vector1+vector2)
print("vector multiplication :- \n",vector1*vector2)
print("Dot product :- \n",np.dot(vector1,vector2))
angle = np.arccos(np.dot(vector1,vector2) / (np.linalg.norm(vector1)*np.linalg.norm(vector2)))
print(angle)

vector addition :- 
 [10 12 12]
vector multiplication :- 
 [16 27 35]
Dot product :- 
 78
0.7078581313546339


In [20]:
resturant_types = np.array(['biriyani','chinese','burger','pizza'])
vectorized_upper = np.vectorize( lambda e : e.upper() )
print(vectorized_upper(resturant_types))

['BIRIYANI' 'CHINESE' 'BURGER' 'PIZZA']


Broadcasting

In [21]:
print(vector1*2)

[ 4  6 14]


Matrix

In [22]:
mat = np.matrix([[1,2,3],[4,5,6],[7,8,9]])
mat2 = np.matrix([[1,2,3],[4,5,6],[7,8,9]])

print(type(mat))

print(mat*mat2)
print(mat+mat2)
print(mat.T)
print(mat.dot(mat2))

<class 'numpy.matrix'>
[[ 30  36  42]
 [ 66  81  96]
 [102 126 150]]
[[ 2  4  6]
 [ 8 10 12]
 [14 16 18]]
[[1 4 7]
 [2 5 8]
 [3 6 9]]
[[ 30  36  42]
 [ 66  81  96]
 [102 126 150]]
